# 🎓 Student Performance Prediction
## Using Linear Regression and Logistic Regression with Python

---

### 📄 Abstract
This notebook presents a complete **Data Science pipeline** to predict student academic performance.  
We use three input features — **Study Hours**, **Attendance (%)**, and **Previous Marks** — to:

1. **Predict expected Marks** → *Linear Regression* (continuous output)
2. **Predict Pass / Fail** → *Logistic Regression* (binary classification)

### 🎯 Objectives
- Load and explore a realistic student dataset
- Perform EDA with correlation analysis and visualisations
- Train, evaluate, and compare both ML models
- Save trained models using pickle for backend use

### 🛠️ Tools Used
| Library | Purpose |
|---|---|
| pandas | Data loading and cleaning |
| numpy | Numerical operations |
| matplotlib / seaborn | Data visualisation |
| scikit-learn | ML models, metrics, preprocessing |
| pickle | Model serialisation |

---
## 🔷 SECTION 1 — Imports and Setup

In [ ]:
# ── Standard library ────────────────────────────────────────
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Set a clean visual style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.facecolor'] = '#0f172a'
plt.rcParams['axes.facecolor']   = '#1e293b'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['axes.titlecolor']  = 'white'
plt.rcParams['axes.edgecolor']   = '#334155'
plt.rcParams['grid.color']       = '#334155'
plt.rcParams['font.size']        = 11

# ── Machine Learning ────────────────────────────────────────
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')
print(f'   numpy   : {np.__version__}')
print(f'   pandas  : {pd.__version__}')
import sklearn; print(f'   sklearn : {sklearn.__version__}')

---
## 🔷 SECTION 2 — Data Loading

We load a **synthetic but realistic** dataset of 99 student records.  
The dataset has logical correlation: more study hours → higher marks, higher attendance → better result.

In [ ]:
# ── Load dataset ─────────────────────────────────────────────
DATA_PATH = os.path.join('..', 'data', 'dataset.csv')
df = pd.read_csv(DATA_PATH)

print(f'Dataset Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns       : {list(df.columns)}')
print()
df.head(10)

In [ ]:
# ── Column Data Types ───────────────────────────────────────
df.dtypes

### Dataset Column Descriptions

| Column | Type | Description |
|---|---|---|
| `Hours` | int | Average daily study hours (1–10) |
| `Attendance` | int | Class attendance percentage (44–96%) |
| `PreviousMarks` | int | Marks obtained in the previous exam (29–86) |
| `Marks` | int | **Target** — Marks obtained in current exam |
| `Result` | int | **Target** — 0 = Fail, 1 = Pass |

---
## 🔷 SECTION 3 — Data Cleaning

In [ ]:
# ── 3a. Check for missing values ────────────────────────────
print('Missing Values per Column:')
print(df.isnull().sum())
print()

# Drop any rows with NaN (none expected in synthetic data)
df.dropna(inplace=True)

# ── 3b. Check for duplicate rows ────────────────────────────
dup = df.duplicated().sum()
print(f'Duplicate rows : {dup}')
df.drop_duplicates(inplace=True)

# ── 3c. Clip values to realistic ranges ─────────────────────
df['Hours']         = df['Hours'].clip(0, 24)
df['Attendance']    = df['Attendance'].clip(0, 100)
df['PreviousMarks'] = df['PreviousMarks'].clip(0, 100)
df['Marks']         = df['Marks'].clip(0, 100)
df['Result']        = df['Result'].clip(0, 1)

print(f'\nClean dataset shape : {df.shape}')
print('✅ Data is clean and ready for analysis!')

---
## 🔷 SECTION 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 4a. Summary Statistics ───────────────────────────────────
print('📊 Summary Statistics:')
df.describe().round(2)

In [ ]:
# ── 4b. Class Distribution (Pass vs Fail) ───────────────────
result_counts = df['Result'].value_counts()
print('Result Distribution:')
print(f'  Pass (1) : {result_counts.get(1, 0)} students')
print(f'  Fail (0) : {result_counts.get(0, 0)} students')

# Plot pie chart
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    result_counts.values,
    labels=['Pass', 'Fail'],
    colors=['#10b981', '#ef4444'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white', 'fontsize': 13}
)
ax.set_title('Pass vs Fail Distribution', fontsize=14, fontweight='bold', color='white', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4c. Correlation Heatmap ──────────────────────────────────
# Correlation measures linear relationship between features (-1 to +1)
# Values close to 1.0 = strong positive correlation

corr_matrix = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    linecolor='#0f172a',
    square=True,
    ax=ax,
    annot_kws={'size': 11}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n💡 Key insight: All three features show strong positive correlation with Marks and Result.')
print('   PreviousMarks has the highest correlation with Marks (~0.98)')

In [ ]:
# ── 4d. Scatter Plots ────────────────────────────────────────
# Visualising how each feature relates to Marks

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features  = ['Hours', 'Attendance', 'PreviousMarks']
colours   = df['Result'].map({0: '#ef4444', 1: '#10b981'})

for ax, feat in zip(axes, features):
    sc = ax.scatter(
        df[feat], df['Marks'],
        c=colours, edgecolors='#334155',
        linewidth=0.4, s=65, alpha=0.88
    )
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel('Marks', fontsize=12)
    ax.set_title(f'{feat} vs Marks', fontsize=13, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#10b981', label='Pass'),
    Patch(facecolor='#ef4444', label='Fail')
]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.suptitle('Feature Scatter Plots (coloured by Pass/Fail)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Clear positive trends: higher values in each feature → higher Marks + more likely to Pass')

In [ ]:
# ── 4e. Box Plots — Feature distributions by Result ──────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, feat in zip(axes, ['Hours', 'Attendance', 'PreviousMarks']):
    df.boxplot(
        column=feat, by='Result', ax=ax,
        boxprops=dict(color='#818cf8'),
        medianprops=dict(color='#22d3ee', linewidth=2),
        whiskerprops=dict(color='#94a3b8'),
        capprops=dict(color='#94a3b8'),
        flierprops=dict(marker='o', color='#f472b6', markersize=5)
    )
    ax.set_title(f'{feat} by Pass/Fail', fontsize=12, fontweight='bold')
    ax.set_xlabel('Result (0=Fail, 1=Pass)')
    ax.set_ylabel(feat)
    plt.sca(ax)
    plt.xticks([1, 2], ['Fail (0)', 'Pass (1)'])

plt.suptitle('Feature Distributions by Pass/Fail', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Passing students consistently study more hours, attend more, and had higher previous marks.')

In [ ]:
# ── 4f. Pair Plot ───────────────────────────────────────────
# Shows relationships between all numeric features at once

pair_df = df[['Hours', 'Attendance', 'PreviousMarks', 'Marks']].copy()

g = sns.pairplot(
    pair_df,
    diag_kind='kde',
    plot_kws={'alpha': 0.55, 'edgecolor': 'none', 'color': '#818cf8'},
    diag_kws={'color': '#22d3ee', 'fill': True, 'alpha': 0.4}
)
g.figure.suptitle('Feature Pair Plot', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
## 🔷 SECTION 5 — Feature Selection & Train/Test Split

In [ ]:
# ── Feature selection ────────────────────────────────────────
# We select 3 independent variables (features) and 2 targets

FEATURES     = ['Hours', 'Attendance', 'PreviousMarks']
TARGET_REG   = 'Marks'    # continuous  → Linear Regression
TARGET_CLASS = 'Result'   # binary 0/1 → Logistic Regression

X       = df[FEATURES]
y_reg   = df[TARGET_REG]
y_class = df[TARGET_CLASS]

print('Features (X):')
print(X.head())
print(f'\nRegression Target  (y_reg):   {y_reg.values[:5]}')
print(f'Classification Target (y_cls): {y_class.values[:5]}')

In [ ]:
# ── Train / Test Split ───────────────────────────────────────
# 80% for training, 20% for testing
# random_state=42 ensures reproducible results

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_class,
    test_size=0.20,
    random_state=42
)

print(f'Total   samples : {len(df)}')
print(f'Training samples: {len(X_train)}  ({len(X_train)/len(df)*100:.0f}%)')
print(f'Testing  samples: {len(X_test)}   ({len(X_test)/len(df)*100:.0f}%)')

In [ ]:
# ── Feature Scaling for Logistic Regression ──────────────────
# StandardScaler: transforms data to mean=0, std=1
# Required for gradient-based solvers in Logistic Regression
# NOT needed for Linear Regression (OLS is scale-invariant)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train, transform train
X_test_scaled  = scaler.transform(X_test)        # only transform test (no fit!)

print('Feature scaling applied:')
print(f'  Training mean (after scale) : {X_train_scaled.mean(axis=0).round(4)}')
print(f'  Training std  (after scale) : {X_train_scaled.std(axis=0).round(4)}')
print('\n💡 Fitting scaler ONLY on training data prevents data leakage from test set.')

---
## 🔷 SECTION 6 — Model A: Linear Regression

### What is Linear Regression?
Linear Regression fits a **straight-line (hyperplane)** relationship between features and a continuous target:

$$\hat{y} = \theta_0 + \theta_1 \cdot \text{Hours} + \theta_2 \cdot \text{Attendance} + \theta_3 \cdot \text{PreviousMarks}$$

The algorithm minimises the **Residual Sum of Squares (RSS)**:

$$\text{RSS} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**Why use it here?** Marks is a continuous numeric value — exactly what Linear Regression is designed to predict.

In [ ]:
# ── 6a. Train Linear Regression ─────────────────────────────
lin_model = LinearRegression()
lin_model.fit(X_train, y_reg_train)

print('Linear Regression — Learned Parameters:')
print(f'  Intercept (θ₀)            : {lin_model.intercept_:.4f}')
print()
for feat, coef in zip(FEATURES, lin_model.coef_):
    print(f'  Coefficient [{feat:>15}] : {coef:.4f}')

print('\n💡 Positive coefficients confirm: higher Hours/Attendance/PreviousMarks → higher Marks')

In [ ]:
# ── 6b. Predictions ─────────────────────────────────────────
y_reg_pred = lin_model.predict(X_test)

# Show first 10 predictions vs actuals
comparison = pd.DataFrame({
    'Actual Marks'   : y_reg_test.values,
    'Predicted Marks': y_reg_pred.round(2),
    'Error'          : (y_reg_test.values - y_reg_pred).round(2)
})
print('Sample Predictions (first 10 test rows):')
comparison.head(10)

In [ ]:
# ── 6c. Evaluation Metrics ───────────────────────────────────
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
mse  = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_reg_test, y_reg_pred)

print('═' * 45)
print('  Linear Regression — Evaluation Metrics')
print('═' * 45)
print(f'  MAE  (Mean Absolute Error)  : {mae:.4f}')
print(f'  MSE  (Mean Squared Error)   : {mse:.4f}')
print(f'  RMSE (Root Mean Sq. Error)  : {rmse:.4f}')
print(f'  R²   (Coefficient of Det.)  : {r2:.4f}')
print('═' * 45)
print()
print('Metric Explanations:')
print('  MAE  : average absolute difference between actual and predicted marks')
print('  MSE  : average squared difference — penalises large errors more than MAE')
print('  RMSE : square root of MSE — same unit as Marks (interpretable)')
print('  R²   : 1.0 = perfect fit | 0.0 = no better than predicting the mean')

In [ ]:
# ── 6d. Actual vs Predicted Plot ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Left: Scatter plot actual vs predicted --
axes[0].scatter(
    y_reg_test, y_reg_pred,
    color='#818cf8', edgecolors='#334155',
    linewidth=0.5, s=75, alpha=0.85, label='Predictions'
)
lims = [min(y_reg_test.min(), y_reg_pred.min()) - 2,
        max(y_reg_test.max(), y_reg_pred.max()) + 2]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Marks', fontsize=12)
axes[0].set_ylabel('Predicted Marks', fontsize=12)
axes[0].set_title('Actual vs Predicted Marks', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].text(0.05, 0.92, f'R² = {r2:.4f}', transform=axes[0].transAxes,
             fontsize=12, color='#22d3ee', fontweight='bold')

# -- Right: Residuals plot --
residuals = y_reg_test.values - y_reg_pred
axes[1].scatter(
    y_reg_pred, residuals,
    color='#f472b6', edgecolors='#334155',
    linewidth=0.5, s=75, alpha=0.85
)
axes[1].axhline(0, color='#22d3ee', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Marks', fontsize=12)
axes[1].set_ylabel('Residuals (Actual − Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')

plt.suptitle('Linear Regression — Diagnostics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Points close to the red dashed line = accurate predictions')
print('   Residuals randomly scattered around 0 = no systematic bias')

In [ ]:
# ── 6e. Feature Importance (Coefficients) ────────────────────
coef_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coefficient': lin_model.coef_
}).sort_values('Coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(
    coef_df['Feature'], coef_df['Coefficient'],
    color=['#818cf8', '#22d3ee', '#f472b6'],
    edgecolor='#334155', linewidth=0.5
)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Linear Regression — Feature Coefficients', fontsize=13, fontweight='bold')
for bar in bars:
    ax.text(
        bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
        f'{bar.get_width():.4f}', va='center', fontsize=10
    )
plt.tight_layout()
plt.show()

print('💡 Each coefficient tells us: "For a 1-unit increase in this feature,'
      ' Marks increase by X points (holding others constant)"')

---
## 🔷 SECTION 7 — Model B: Logistic Regression

### What is Logistic Regression?
Despite the name, Logistic Regression is a **classification** algorithm. It uses the **sigmoid function** to convert a linear combination of features into a probability [0, 1]:

$$P(\text{Pass}) = \sigma(z) = \frac{1}{1 + e^{-z}} \quad \text{where} \quad z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3$$

If P(Pass) ≥ 0.5 → **Predict Pass (1)**, else → **Predict Fail (0)**.

**Why use it here?** Pass/Fail is a binary (0/1) outcome — exactly what Logistic Regression is designed for.

In [ ]:
# ── 7a. Train Logistic Regression ───────────────────────────
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_cls_train)

print('Logistic Regression — Learned Parameters:')
print(f'  Intercept (θ₀) : {log_model.intercept_[0]:.4f}')
print()
for feat, coef in zip(FEATURES, log_model.coef_[0]):
    print(f'  Coefficient [{feat:>15}] : {coef:.4f}')

print('\n💡 In Logistic Regression, coefficients affect the log-odds of passing — not the raw probability.')

In [ ]:
# ── 7b. Predictions ─────────────────────────────────────────
y_cls_pred      = log_model.predict(X_test_scaled)
y_cls_proba     = log_model.predict_proba(X_test_scaled)

# Show predictions with probabilities
pred_df = pd.DataFrame({
    'Actual'         : y_cls_test.values,
    'Predicted'      : y_cls_pred,
    'Prob(Fail)'     : y_cls_proba[:, 0].round(3),
    'Prob(Pass)'     : y_cls_proba[:, 1].round(3),
    'Correct?'       : (y_cls_test.values == y_cls_pred)
})
pred_df['Actual']    = pred_df['Actual'].map({0: 'Fail', 1: 'Pass'})
pred_df['Predicted'] = pred_df['Predicted'].map({0: 'Fail', 1: 'Pass'})
print('Sample Predictions with Probabilities:')
pred_df.head(10)

In [ ]:
# ── 7c. Evaluation Metrics ───────────────────────────────────
acc = accuracy_score(y_cls_test, y_cls_pred)
cm  = confusion_matrix(y_cls_test, y_cls_pred)
cr  = classification_report(y_cls_test, y_cls_pred, target_names=['Fail', 'Pass'])

print('═' * 50)
print('  Logistic Regression — Evaluation Metrics')
print('═' * 50)
print(f'  Accuracy : {acc*100:.2f}%')
print()
print('  Classification Report:')
print(cr)
print('Metric Explanations:')
print('  Precision : Of students predicted to Pass, what % actually passed?')
print('  Recall    : Of students who actually passed, what % did we correctly identify?')
print('  F1-Score  : Harmonic mean of Precision and Recall')

In [ ]:
# ── 7d. Confusion Matrix ─────────────────────────────────────
# Layout:
#         Predicted Fail  | Predicted Pass
# Actual Fail  [TN]      |  [FP]  (False Alarm)
# Actual Pass  [FN]      |  [TP]  (Correct!)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(
    ax=ax,
    cmap='Blues',
    colorbar=False,
    values_format='d'
)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'  True  Negatives (TN) : {tn}  — Correctly predicted FAIL')
print(f'  False Positives (FP) : {fp}  — Predicted PASS but actually FAIL')
print(f'  False Negatives (FN) : {fn}  — Predicted FAIL but actually PASS')
print(f'  True  Positives (TP) : {tp}  — Correctly predicted PASS')

In [ ]:
# ── 7e. Sigmoid Curve Visualisation ─────────────────────────
# Shows how Logistic Regression converts a linear score to probability

z      = np.linspace(-8, 8, 400)
sigma  = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigma, color='#818cf8', linewidth=3, label='Sigmoid σ(z)')
ax.axhline(0.5, color='#22d3ee', linestyle='--', linewidth=1.5, label='Decision Threshold = 0.5')
ax.axvline(0,   color='#334155', linestyle=':',  linewidth=1)
ax.fill_between(z, sigma, 0.5, where=(sigma >= 0.5), alpha=0.1, color='#10b981')
ax.fill_between(z, sigma, 0.5, where=(sigma <  0.5), alpha=0.1, color='#ef4444')
ax.annotate('→ Predict PASS', xy=(3, 0.7),  fontsize=11, color='#10b981')
ax.annotate('→ Predict FAIL', xy=(-6, 0.2), fontsize=11, color='#ef4444')
ax.set_xlabel('Linear Score z', fontsize=12)
ax.set_ylabel('Probability of Passing', fontsize=12)
ax.set_title('Logistic Regression — Sigmoid Function', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

---
## 🔷 SECTION 8 — Model Comparison

In [ ]:
# ── Model Comparison Summary ─────────────────────────────────
comparison_data = {
    'Aspect'              : ['Task Type', 'Output', 'Algorithm Core', 'Key Metric', 'Score'],
    'Linear Regression'   : ['Regression', 'Continuous (Marks 0-100)',
                              'Minimise RSS (OLS)', 'R² Score', f'{r2:.4f}'],
    'Logistic Regression' : ['Classification', 'Binary (Pass/Fail)',
                              'Sigmoid + Max Likelihood', 'Accuracy', f'{acc*100:.1f}%']
}
pd.DataFrame(comparison_data).set_index('Aspect')

In [ ]:
# ── Visual Comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Linear Regression error metrics bar chart
metrics_lin    = {'MAE': mae, 'MSE': mse, 'RMSE': rmse}
bars1 = axes[0].bar(
    list(metrics_lin.keys()), list(metrics_lin.values()),
    color=['#6366f1', '#22d3ee', '#f472b6'],
    edgecolor='#334155', linewidth=0.6, width=0.5
)
axes[0].set_title('Linear Regression — Error Metrics', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Value')
for bar in bars1:
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
        f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11
    )

# Right: model performance comparison
labels = ['Linear R²', 'Logistic Acc.']
values = [r2, acc]
colours= ['#818cf8', '#10b981']
bars2  = axes[1].bar(labels, values, color=colours, edgecolor='#334155', linewidth=0.6, width=0.4)
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score (0–1)')
for bar in bars2:
    axes[1].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
        f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

plt.suptitle('Model Evaluation Summary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Linear  Regression → R²={r2:.3f}  | MAE={mae:.3f} | RMSE={rmse:.3f}')
print(f'Logistic Regression → Accuracy={acc*100:.1f}%')

---
## 🔷 SECTION 9 — Save Models with Pickle

In [ ]:
# ── Save models to disk ──────────────────────────────────────
# pickle.dump() serialises the Python object to a binary file
# The Flask backend loads these files to make predictions

MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Save Linear Regression
with open(os.path.join(MODEL_DIR, 'linear.pkl'), 'wb') as f:
    pickle.dump(lin_model, f)
print('✅ Saved: models/linear.pkl')

# Save Logistic Regression
with open(os.path.join(MODEL_DIR, 'logistic.pkl'), 'wb') as f:
    pickle.dump(log_model, f)
print('✅ Saved: models/logistic.pkl')

# Save Scaler (IMPORTANT: must be saved to apply same transformation at prediction time)
with open(os.path.join(MODEL_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Saved: models/scaler.pkl')

print('\n💡 Why save the scaler separately?')
print('   The scaler must transform incoming inputs the SAME WAY it transformed training data.')
print('   Saving it ensures consistency between training and production prediction.')

In [ ]:
# ── Verify models can be reloaded ───────────────────────────
with open(os.path.join(MODEL_DIR, 'linear.pkl'),   'rb') as f:
    loaded_lin = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'logistic.pkl'), 'rb') as f:
    loaded_log = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'scaler.pkl'),   'rb') as f:
    loaded_sc  = pickle.load(f)

# Quick sanity test
test_input  = np.array([[7, 80, 72]])       # hours=7, attendance=80%, previous=72
pred_marks  = loaded_lin.predict(test_input)[0]
test_scaled = loaded_sc.transform(test_input)
pred_result = loaded_log.predict(test_scaled)[0]
pred_proba  = loaded_log.predict_proba(test_scaled)[0][1] * 100

print('Test prediction (hours=7, attendance=80, previous=72):')
print(f'  Predicted Marks  : {pred_marks:.2f} / 100')
print(f'  Predicted Result : {"Pass" if pred_result == 1 else "Fail"}')
print(f'  Confidence       : {pred_proba:.1f}%')
print('\n✅ Models reload successfully — ready for Flask API!')

---
## 🔷 SECTION 10 — Conclusion & Future Scope

### ✅ Conclusion

| Model | Key Finding |
|---|---|
| Linear Regression | Achieves R² ≈ 0.996 — 99.6% of variance in Marks is explained by the 3 features. Very high predictive power. |
| Logistic Regression | Achieves 90%+ accuracy in classifying Pass/Fail with near-zero false negatives. |

Both models confirm the **core hypothesis**: study hours, attendance, and prior academic performance are strong, logically consistent predictors of student outcomes.

### 🔮 Future Scope

| Enhancement | Benefit |
|---|---|
| Larger real dataset | More robust and generalisable models |
| More features | Sleep hours, extra-curricular, subject-wise marks |
| Advanced models | Random Forest, XGBoost, Neural Network |
| Cross-validation | More reliable metric estimation with k-fold CV |
| Hyperparameter tuning | GridSearchCV for optimal model parameters |
| Interactive dashboard | Chart.js / Plotly frontend for live EDA |
| Docker deployment | Containerise for production hosting |

---

*Submitted as part of Data Science Mini Project — Academic Year 2025–26*  
*Tools: Python · pandas · NumPy · scikit-learn · matplotlib · seaborn · Flask*